# 📘 Notebook 18: Measuring Scale

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 6 of 6 in this module · (18 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Estimate how much memory a table needs, and how many rows one machine can hold
- Run one analytical question in **pandas**, **SQLite** and **Spark** on growing copies of the same data, and time each fairly
- Separate a tool's **fixed overhead** from its **cost per row**, and find where the lines cross
- Explain why a cluster engine loses on small data, and what it offers when the data grows
- Turn the measurements into a reasoned recommendation, the kind a platform evaluation needs

> **Prerequisites:** Notebooks 07, 09, 10 and 12 (memory limits, Spark, when a cluster is the wrong tool, the cost of the cloud). Notebooks 14 to 16 (this warehouse).
>
> 🔭 **Why this notebook matters:** every claim about *performance at scale* deserves a measurement, and this course has made the same point many times: the heavy machinery is not always faster. Here we settle it for the AdventureWorks warehouse itself. We make the sales table 1, 8, 64 and 256 times larger, ask the same question of three tools, and plot the results. The plot is small, but the way of thinking behind it is how platform decisions should be made, and it is the evidence a report on *performance at scale* or *cost efficiency* ought to contain.

> 📦 **Libraries used in this notebook.** **pandas**, **SQLite**, **PySpark** (local mode), **PyArrow** for Parquet files (Notebook 05), **psutil** to read the machine's memory (Notebook 12), and **Matplotlib**. The cell below installs what is missing; in Google Colab everything is already present.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

In [ ]:
%pip install -q pyspark pyarrow psutil

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

In [ ]:
import gc
import sqlite3
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import psutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
print(f"This machine: {psutil.cpu_count()} cores, {psutil.virtual_memory().total / 1e9:.1f} GB of memory")

## 1. How big is the data, really?

### Intuition first
The first question of Notebook 12 was: *how big is the data, really?* Before choosing a tool, measure the bytes. A table in memory costs a certain number of bytes per row; divide the machine's memory by it, keep a safety margin, and you know roughly how many rows one machine can work with.

In [ ]:
sales = load("FactInternetSales")
per_row = sales.memory_usage(deep=True).sum() / len(sales)
print(f"FactInternetSales: {len(sales)} rows, {per_row:.0f} bytes per row in pandas, all 26 columns")

slim = pd.DataFrame({
    "year": pd.to_datetime(sales["OrderDate"]).dt.year.astype("int16"),
    "product_key": sales["ProductKey"].astype("int32"),
    "customer_key": sales["CustomerKey"].astype("int32"),
    "revenue": sales["SalesAmount"].astype("float64"),
})
slim_row = slim.memory_usage(deep=True).sum() / len(slim)
print(f"Only the four columns we need, with compact types: {slim_row:.0f} bytes per row")

usable = psutil.virtual_memory().total * 0.5          # leave half for the system and for intermediate results
for label, size in [("all columns", per_row), ("four compact columns", slim_row)]:
    print(f"{label:<22} about {usable / size / 1e6:,.0f} million rows fit in half of this machine's memory")

Choosing columns and types is worth more than any engine: the slim table needs a small fraction of the memory per row. At the size AdventureWorks has today, its sales would have to grow by a factor of hundreds before one ordinary machine struggled. Keep that factor in mind while the experiment runs.

## 2. The experiment

### Intuition first
One question, typical of a warehouse: **revenue and number of distinct customers by year and product category**. It needs a join (sales to category) and a grouping with a distinct count, which is the expensive kind (Notebook 08). We make copies of the sales: *k* copies, each with its own customer numbers, so that the distinct counts grow too. Then each tool answers the same question on the same rows, and we time **only the question**, not the loading, which each tool does differently.

To be fair to Spark, it reads the data from a Parquet file, as it would on a cluster, and we run each query once before timing it, so that start-up costs are not counted against it.

In [ ]:
category = (load("DimProduct")[["ProductKey", "ProductSubcategoryKey"]]
            .merge(load("DimProductSubcategory")[["ProductSubcategoryKey", "ProductCategoryKey"]], on="ProductSubcategoryKey")
            .merge(load("DimProductCategory")[["ProductCategoryKey", "EnglishProductCategoryName"]], on="ProductCategoryKey")
            [["ProductKey", "EnglishProductCategoryName"]]
            .rename(columns={"ProductKey": "product_key", "EnglishProductCategoryName": "category"}))

def copies(k):
    """k copies of the slim sales table; each copy gets its own customers."""
    parts = [slim.assign(customer_key=slim["customer_key"] + i * 100_000) for i in range(k)]
    return pd.concat(parts, ignore_index=True)

def best_of(function, repeats=3):
    times = []
    for i in range(repeats):
        started = time.perf_counter()
        result = function()
        times.append(time.perf_counter() - started)
    return result, min(times)

The three versions of the question:

In [ ]:
def ask_pandas(df):
    return (df.merge(category, on="product_key")
              .groupby(["year", "category"])
              .agg(revenue=("revenue", "sum"), customers=("customer_key", "nunique")))

QUESTION_SQL = '''
    SELECT s.year, c.category, SUM(s.revenue) AS revenue, COUNT(DISTINCT s.customer_key) AS customers
    FROM sales AS s JOIN category AS c ON c.product_key = s.product_key
    GROUP BY s.year, c.category'''

def ask_spark(sdf, scat):
    return (sdf.join(F.broadcast(scat), "product_key")
               .groupBy("year", "category")
               .agg(F.sum("revenue").alias("revenue"), F.countDistinct("customer_key").alias("customers"))
               .collect())

In [ ]:
spark = (SparkSession.builder.master("local[*]").appName("measuring-scale")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
scat = spark.createDataFrame(category)

results = []
for k in [1, 8, 64, 256]:
    df = copies(k)

    answer_pd, t_pandas = best_of(lambda: ask_pandas(df))

    db = sqlite3.connect(":memory:")
    df.to_sql("sales", db, index=False)
    category.to_sql("category", db, index=False)
    answer_sql, t_sqlite = best_of(lambda: db.execute(QUESTION_SQL).fetchall(), repeats=1)
    db.close()

    df.to_parquet("sales_copies.parquet", index=False)
    sdf = spark.read.parquet("sales_copies.parquet")
    ask_spark(sdf, scat)                                      # warm-up run, not timed
    answer_spark, t_spark = best_of(lambda: ask_spark(sdf, scat))

    totals = [answer_pd["revenue"].sum(), sum(r["revenue"] for r in answer_spark), sum(r[2] for r in answer_sql)]
    same = bool(np.allclose(totals, totals[0], rtol=1e-9))     # sums of floats differ in the last digits
    results.append({"copies": k, "rows": len(df), "pandas_s": t_pandas, "sqlite_s": t_sqlite,
                    "spark_s": t_spark, "same_answer": same})
    print(f"{k:>4} copies, {len(df):>11,} rows:  pandas {t_pandas:6.2f} s   SQLite {t_sqlite:6.2f} s   "
          f"Spark {t_spark:6.2f} s   same answer: {same}")
    del df, sdf
    gc.collect()

results = pd.DataFrame(results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for column, label in [("pandas_s", "pandas"), ("sqlite_s", "SQLite"), ("spark_s", "Spark (local)")]:
    ax.plot(results["rows"], results[column], marker="o", label=label)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("rows in the sales table")
ax.set_ylabel("seconds for the question (log scale)")
ax.set_title("One question, three tools, growing data")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

## 3. Reading the plot

### Intuition first
Every tool's time is roughly **a fixed overhead plus a cost per row**. On a log-log plot, a tool whose time is all per-row cost climbs in a straight line; a tool dominated by overhead starts high and flat. Fit the two numbers to the measurements and the plot becomes an explanation.

In [ ]:
for column in ["pandas_s", "sqlite_s", "spark_s"]:
    per_row, overhead = np.polyfit(results["rows"], results[column], 1)
    print(f"{column[:-2]:<7} overhead about {max(overhead, 0):5.2f} s,  about {per_row * 1e6:6.2f} s per million rows")

Typical results, and the reasons behind them:

- **pandas** has almost no overhead and works in memory with vectorised code. On data that fits, it is very hard to beat, and it stays ahead here at every size that fitted.
- **SQLite** is a single-threaded row store: it reads every column of every row and sorts for the distinct count. Its cost per row is the highest, so it falls behind as the data grows. Its strengths, transactions, indexes, a file on disk, are not what this question needs.
- **Spark** pays a fixed price on every query: planning, scheduling tasks, moving data between stages. On small data that price is the whole cost. But its cost per row is spread across all the cores, and it reads only the columns it needs from Parquet, so its line rises slowly.

Whether Spark actually overtakes pandas on one machine depends on the machine and the question, and on many laptops it never does while the data still fits in memory. That is the result of Notebook 10, now on a real warehouse.

## 4. Where one machine stops

Spark's real argument is not speed on one machine. It is what happens when the data does **not** fit. pandas needs all the rows in memory at once; Spark processes them partition by partition, can spill to disk, and can spread the partitions over many machines. Combine the bytes per row of Section 1 with the measured cost per row, and estimate the limits of this machine:

In [ ]:
per_row_pd = np.polyfit(results["rows"], results["pandas_s"], 1)[0]
limit_rows = usable / slim_row
print(f"pandas, slim table: about {limit_rows / 1e6:,.0f} million rows fit; the question would then take "
      f"about {per_row_pd * limit_rows:,.0f} s.")
print(f"That is about {limit_rows / len(slim):,.0f} times today's AdventureWorks sales.")

Beyond that point there are three choices, in order of cost and complexity:

1. **Make the data smaller**: fewer columns, compact types, Parquet, a summary table (Notebooks 05 and 14).
2. **Process it in pieces on one machine**: streaming through files, as in Notebook 07, or a single-machine engine that does it for you.
3. **Use a cluster**: Spark on many machines, or a cloud warehouse that hides the cluster (Redshift, Synapse, BigQuery). You gain capacity and pay in money, in start-up time, and in complexity (Notebook 12).

For a data set the size of AdventureWorks, the honest recommendation is option 1 on one machine, and that is a perfectly good answer to give, provided it comes with the measurements that justify it and a statement of the growth at which the answer would change.

## 5. Where we stand

| Size of the data | Sensible tool | Why |
|---|---|---|
| Fits in memory many times over (like AdventureWorks) | pandas, or SQL in a single-file database | no overhead; simplest to run and to explain |
| Fits on one machine's disk, not in its memory | stream through files, Parquet, a single-machine engine | avoids a cluster's cost and complexity |
| Larger than one machine, or growing fast | Spark on a cluster, or a cloud warehouse | capacity, parallelism, fault tolerance (Notebooks 08 to 10) |

Module E took one real warehouse through the whole course: it was profiled (13), built with keys, views and indexes (14), reshaped into documents (15), explored with Spark (16), used to train and evaluate models (17), and finally measured (18). The tools changed at every step; the method did not. Measure the data, state the assumptions, choose the simplest tool that is enough, and know at what size that choice stops being right.

---
## ✏️ Exercises

### Exercise 1 (Parquet for pandas too)
Spark read Parquet; pandas worked on a table already in memory. Time how long pandas takes to **read** `sales_copies.parquet` for 64 copies, first all columns and then only `year` and `revenue`, and compare with reading the same rows from CSV.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
df64 = copies(64)
df64.to_parquet("sales_copies.parquet", index=False)
df64.to_csv("sales_copies.csv", index=False)
_, t_all = best_of(lambda: pd.read_parquet("sales_copies.parquet"), repeats=1)
_, t_two = best_of(lambda: pd.read_parquet("sales_copies.parquet", columns=["year", "revenue"]), repeats=1)
_, t_csv = best_of(lambda: pd.read_csv("sales_copies.csv"), repeats=1)
print(f"Parquet, all columns: {t_all:.2f} s   Parquet, two columns: {t_two:.2f} s   CSV: {t_csv:.2f} s")
print(f"sizes on disk: Parquet {os.path.getsize('sales_copies.parquet') / 1e6:.0f} MB, "
      f"CSV {os.path.getsize('sales_copies.csv') / 1e6:.0f} MB")
```

*Loading is often slower than the question itself, and the file format decides it. This is Notebook 05's lesson measured on the warehouse: on a cloud service that charges per byte read, the same ratio becomes the bill.*

</details>

### Exercise 2 (A cheaper question)
Remove the distinct count: time only `SUM(revenue)` by year and category in pandas and Spark at 64 copies. Does the gap between the tools change? Why?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
df64 = copies(64)
_, t_pd = best_of(lambda: df64.merge(category, on="product_key").groupby(["year", "category"])["revenue"].sum())
df64.to_parquet("sales_copies.parquet", index=False)
s64 = spark.read.parquet("sales_copies.parquet")
cheap = lambda: s64.join(F.broadcast(scat), "product_key").groupBy("year", "category").agg(F.sum("revenue")).collect()
cheap()
_, t_sp = best_of(cheap)
print(f"sum only, 64 copies:  pandas {t_pd:.2f} s   Spark {t_sp:.2f} s")
print(results.loc[results['copies'] == 64, ['pandas_s', 'spark_s']])
```

*A distinct count has to remember every customer seen, and in Spark it needs an extra shuffle; a sum needs one number per group. The cheaper the question, the larger the share of Spark's fixed overhead in its time.*

</details>

### Exercise 3 (Partitions)
Run the Spark question at 64 copies with the file read as 1, 8 and 32 partitions (`sdf.repartition(n)` before the question, cached and counted first). What happens, and why?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
df64 = copies(64)
df64.to_parquet("sales_copies.parquet", index=False)
for n in (1, 8, 32):
    part = spark.read.parquet("sales_copies.parquet").repartition(n).cache()
    part.count()
    _, t = best_of(lambda: ask_spark(part, scat))
    print(f"{n:>3} partitions: {t:.2f} s")
    part.unpersist()
```

*Compare the times with Spark's fixed overhead from Section 3. Where the overhead dominates, as it does on small data and on some machines, the number of partitions hardly matters, which is this notebook's lesson again. Where the work dominates, one partition leaves all but one core idle, one to a few partitions per core use the machine fully, and many more add scheduling cost for little gain. On a cluster the same reasoning applies to every machine.*

</details>

### Exercise 4 (A recommendation in numbers, a little harder)
Write a short function `recommend(rows, bytes_per_row, machine_gb)` that returns `"pandas on one machine"`, `"one machine, in pieces"` or `"a cluster"`, using: fits in half the memory, fits in ten times the memory (on disk), or more. Apply it to today's sales and to 1,000 and 100,000 times today's sales, on a 16 GB machine.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def recommend(rows, bytes_per_row, machine_gb):
    need_gb = rows * bytes_per_row / 1e9
    if need_gb <= machine_gb * 0.5:
        return "pandas on one machine"
    if need_gb <= machine_gb * 10:
        return "one machine, in pieces"
    return "a cluster"

for growth in (1, 1_000, 100_000):
    rows = len(slim) * growth
    print(f"{growth:>7}x today ({rows:>14,} rows): {recommend(rows, slim_row, 16)}")
```

*Crude, but explicit: anyone can check the assumptions and the arithmetic, and change them. That is what makes a recommendation defensible, which a sentence such as 'Spark is better for big data' is not.*

</details>

## 🔑 Key takeaways

- Measure the **bytes per row** first; choosing columns and types often matters more than choosing an engine.
- Time **only the question**, warm up first, and check that every tool returns the **same answer**.
- A tool's time is a **fixed overhead** plus a **cost per row**; on small data the overhead decides.
- On data that fits in memory, pandas usually wins; Spark's case is data that does **not** fit, and capacity across machines.
- A defensible recommendation states the **measurements**, the **assumptions**, and the **size at which it would change**.

### 🎓 Congratulations: you have completed Module E, and the whole course!

You took one real data warehouse through every idea of the course: you profiled it, built it with the rules that keep it true, reshaped it into documents, explored it and trained models on it with Spark, and finally measured what scale would mean for it. On the way the measurements kept their habit of surprising: the simplest tool often won, and the reasons why it won were more instructive than the result.

The questions to carry with you are the same as at the end of Module D, now answered on real data. How big is the data, really? How is it read, and how does it change? What must never be wrong? And what, exactly, am I giving up to get this?

In [ ]:
spark.stop()

---
*End of Module E: and the end of the course.*

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*